In [ ]:
!pip install -q -U langchain
!pip install -q -U langchain-community
!pip install -q faiss-cpu
!pip install -q -U huggingface_hub
!pip install -q -U openai
!pip install -q -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 810.8/810.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 2.2 MB/s eta 0:00:00


In [ ]:
!pip show sentence-transformers

Name: sentence-transformers
Version: 5.1.0
Summary: Embeddings, Retrieval, and Reranking
Home-page: https://www.SBERT.net
Author: 
Author-email: Nils Reimers <info@nils-reimers.de>, Tom Aarsen <tom.aarsen@huggingface.co>
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, Pillow, scikit-learn, scipy, torch, tqdm, transformers, typing_extensions
Required-by: 


In [ ]:
import json
from langchain.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

In [ ]:
with open('chatbot_training.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

In [ ]:
data[:2]

[{'Transcript': "[Student] Sorry I'm late. I just like, totally forgot. \n\n[Teacher] That's okay. Uh, it happens. Um, yeah. I'm glad you guys made it. Uh, should I expect Brody at some point too? \n\n[Student] Yeah. \n\n[Teacher] All right, cool. Uh, yeah, no problem.\n",
  'Observation': "0:00 - People are entering the Zoom link, and say sorry that they're late.",
  'Feedback': "Nice job being accepting and inviting by saying it's not a problem that they're late.  Try to talk to them a bit and ask them how they're doing.  Also, it's completely fine to ask them to turn on their videos so you can see them!  This may help to keep them more engaged.",
  'Teacher': 'Alex'},
 {'Transcript': "[Teacher] So we're gonna\nlook at what, 2019 and then maybe, uh, 2018. Also, just the last few problems in each one. Some of these might be familiar, I'm not sure. Um, and if you like totally remember how to do it, then just let me know. But even if it's familiar, um, it might be worth going over again

In [ ]:
train_data = []
for item in data:
  if item['Teacher'] != 'Alex' and item['Teacher'] != 'Zavia1':
    train_data.append(item)

train_data[:2]

[{'Transcript': '[Teacher][00:00:00] [00:01:00] [00:02:00] [00:03:00] [00:04:00] [00:05:00]\n[00:06:00] [00:07:00] [00:08:00] Problem so far.',
  'Observation': "8:34: The first talking I heard was Da'ron asking, “How are you doing with this problem so far?”",
  'Feedback': "“How is it going” or “how are you doing” is a very vague question.  The answer to that question is literally, “I'm doing well,” or “not so good.”  But this doesn't give you anything to go on.  A better question is a more specific one, such as, “what strategy are you using?” or “What is an estimate of the answer?”\n",
  'Teacher': "Da'ron"},
 {'Transcript': "[Student] Um, I\nthink I'm getting closer to the answer, but I thought I had to answer, but my calculations were off. I guess I'm kind of just trying error,\n\n[Teacher] uh, I can't really hear what you're saying too. \n\n[Student] Well,\noh. I'm, I just trying [00:09:00] random numbers to see if it works.\n\n[Teacher] Okay. Uh, okay. How, how would you, how do 

In [ ]:
documents = []
for item in train_data:
    text = f"Transcript:\n{item['Transcript']}\nObservation:\n{item['Observation']}\nFeedback:\n{item['Feedback']}"
    documents.append(text)

documents[:2]

["Transcript:\n[Teacher][00:00:00] [00:01:00] [00:02:00] [00:03:00] [00:04:00] [00:05:00]\n[00:06:00] [00:07:00] [00:08:00] Problem so far.\nObservation:\n8:34: The first talking I heard was Da'ron asking, “How are you doing with this problem so far?”\nFeedback:\n“How is it going” or “how are you doing” is a very vague question.  The answer to that question is literally, “I'm doing well,” or “not so good.”  But this doesn't give you anything to go on.  A better question is a more specific one, such as, “what strategy are you using?” or “What is an estimate of the answer?”\n",
 "Transcript:\n[Student] Um, I\nthink I'm getting closer to the answer, but I thought I had to answer, but my calculations were off. I guess I'm kind of just trying error,\n\n[Teacher] uh, I can't really hear what you're saying too. \n\n[Student] Well,\noh. I'm, I just trying [00:09:00] random numbers to see if it works.\n\n[Teacher] Okay. Uh, okay. How, how would you, how do you find the area, right, or\nsaying y

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_texts(documents, embeddings)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
    max_tokens=1000,
    openai_api_key=""
)

In [ ]:
# retriever.get_relevant_documents("""
#   [00:00:00] Rename myself.
# Sorry I'm late. I just like, totally forgot. That's okay. Uh, it happens. Um, yeah. I'm glad you guys made it. Uh, should I expect Brody at some point too? Yeah. All right, cool. Uh, yeah, no problem. So we're gonna look at what, 2019 and then maybe, uh, 2018. Also, just the last few problems in each one. Some of these might be familiar, I'm not sure.
# Um, and if you like totally remember how to do it, then just let me know. But even if it's familiar, um, it might be worth going over again. 'cause these are hard. So for now, just look at 22 and 23. Um, and lemme know how [00:01:00] that goes.[00:02:00] [00:03:00] [00:04:00]
# Hey, birdie, uh, you can just, uh, hop in with us, uh, 22 and 23 right now. Okay.[00:05:00] [00:06:00] [00:07:00] [00:08:00]
# How are these two problems going for you guys? Uh, I'm trying to solve 22. Oh Algebra Clear. So 22, I'm on 23. Okay. 22 I finished and then 23, I'm almost done. Alright, cool. Uh, let's give it a couple minutes. Uh, and if you're stuck, uh, please feel free to say something.[00:09:00] [00:10:00] [00:11:00]
# All right. Are you feeling a little bit more prepared to look at these two together? Uh, I think I got 22, but I'm kind of stuck on 23. Okay. Um, Aiden and Brody, how about you?

# """)

In [ ]:
prompt_template = """
You are an experienced and friendly mentor for Orlando Math Circle facilitators. Your goal is to offer supportive, down-to-earth, and actionable suggestions based on a transcript of an online session.

**Session Context:**
- **Session Format:** {session_format}
- **Number of Students:** {num_students}
- **Grade Level:** {grade_level}
- **Session Topic:** {session_topic}

Your feedback should be guided by the core principles of inquiry-based learning: encouraging exploration, promoting independence, building confidence, asking open-ended questions, being less helpful (guiding instead of telling), and fostering collaboration.

**Instructions for Feedback:**
1.  Analyze the provided transcript, keeping the **Session Context** in mind.
2.  Identify specific moments (reference timestamps if available) that are working well and areas where there are opportunities for growth.
3.  Structure your feedback with clear headings for different teaching moments or themes.
4.  For each point, provide a direct quote or a description of the event, explain your thoughts on it, and offer a concrete suggestion. Use "Observation," "Feedback," and "Suggestion" as the subheadings for each point.
5.  **Crucially, treat each problem number (e.g., 'problem 22', 'problem 23') as a distinct, unrelated challenge.** Since the session topic is AMC Exam Problems, you should assume problems are not thematically connected and increase in difficulty. Do not suggest that strategies for one will apply to the other unless a student explicitly makes that connection.
6.  **Maintain a supportive, conversational, and down-to-earth tone.** Imagine you are a helpful colleague, not a formal evaluator. Do not explicitly name the core principles in your response.

Here are some high-quality examples of feedback based on past transcripts:
{examples}

Now, analyze the following new transcript and provide your feedback:
{transcript}

Your expert feedback:
"""

In [ ]:
def generate_feedback_with_rag(transcript_text, session_format, num_students, grade_level, session_topic):
    retrieved_docs = retriever.get_relevant_documents(transcript_text)
    examples_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # Format the prompt with all the necessary variables
    prompt = prompt_template.format(
        session_format=session_format,
        num_students=num_students,
        grade_level=grade_level,
        session_topic=session_topic,
        examples=examples_text,
        transcript=transcript_text
    )

    # Call GPT-4o-mini
    response = llm.invoke(prompt)
    return response.content

In [ ]:
!pip install python-docx
from docx import Document
import textwrap

In [ ]:
def load_transcript(docx_path):
  doc = Document(docx_path)
  return "\n".join([p.text for p in doc.paragraphs if p.text.strip()])

In [ ]:
transcript_text = load_transcript("medium_alex_transcript.docx")
print(transcript_text)

[Student] Sorry I'm late. I just like, totally forgot.
[Teacher] That's okay. Uh, it happens. Um, yeah. I'm glad you guys made it. Uh, should I expect Brody at some point too?
[Student] Yeah.
[Teacher] All right, cool. Uh, yeah, no problem.
[Teacher] So we're gonna
look at what, 2019 and then maybe, uh, 2018. Also, just the last few problems in each one. Some of these might be familiar, I'm not sure. Um, and if you like totally remember how to do it, then just let me know. But even if it's familiar, um, it might be worth going over again. 'Cause these are hard. So for now, just look at 22 and 23. Um, and lemme know
how that goes.
[00:05:00] [00:06:00] [00:07:00] [00:08:00]
[Teacher] How are these two problems going for you guys?
[Student] Uh, I'm trying to solve 22. Oh Algebra Clear. So 22, I'm on 23. Okay. 22 I finished and then 23,
I'm almost done.
[Teacher] Alright, cool. Uh, let's give it a couple minutes. Uh, and if you're stuck, uh, please feel free to say something.[00:09:00]
[0

In [ ]:
# 1. define the context for the session
session_format = "Online via Zoom"
num_students = 3
grade_level = "High School"
session_topic = "AMC Exam Problems"

#2. Load the transcript
# (Ensure you have created a file named "medium_alex_transcript.docx" and uploaded it)
transcript_text = load_transcript("medium_alex_transcript.docx")

# 3.Call the function with the transcript and the new context
feedback = generate_feedback_with_rag(
    transcript_text,
    session_format=session_format,
    num_students=num_students,
    grade_level=grade_level,
    session_topic=session_topic
)

# 4. Print the feedback
print(feedback)

### Opening and Setting the Stage

**Observation:**
At the beginning of the session, you welcomed the students warmly and acknowledged their arrival, saying, “That’s okay. Uh, it happens. Um, yeah. I’m glad you guys made it.” 

**Feedback:**
This is a great way to create a supportive environment. Acknowledging their presence and being understanding about tardiness helps students feel comfortable and valued.

**Suggestion:**
Consider incorporating a brief icebreaker or check-in question at the start of each session. This could help build rapport and encourage students to share their thoughts or feelings about math or the problems they are tackling.

---

### Problem Selection and Engagement

**Observation:**
You mentioned, “So for now, just look at 22 and 23. Um, and lemme know how that goes.” 

**Feedback:**
This approach encourages students to engage with the problems independently before discussing them as a group. It allows them to explore their thought processes and strategies.

**